# 3. Inline & Spill, Chain of Recall와 세션 로그

근거: [설계 글 §3.3](https://x.com/DeepSeekHarness/status/2116857257245987236?s=20), 저장소 docs/architecture.md의 Session log.

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## 문맥 창을 제한된 작업 공간으로 보기
자주 곧 쓸 요약·연관 ID는 inline, 긴 본문은 외부 저장소로 spill합니다. 단순 절단과 달리 원문으로 돌아갈 참조가 남아야 합니다. Chain of Recall은 현재 사건 → Thread → Worker → 다른 Thread처럼 **명시적 참조**를 따라가는 회수입니다. 벡터 검색과 동일하지 않으며 둘은 함께 쓸 수 있습니다.

아래 길이 예산은 Python 문자 수입니다. 모델 token 예산과 같지 않습니다. 실서비스는 해당 tokenizer와 권한 검사를 사용해야 합니다.

In [1]:
from langchain.tools import tool
objects={
 'im:42':{'text':'IM에서 정한 목표: Plan A로 버그 수정. '*20,'links':['worker:7']},
 'worker:7':{'text':'인증 버그 수정 담당','links':['pr:9']},
 'pr:9':{'text':'PR 검토에서 Plan B로 변경. 회귀 테스트 추가.','links':[]},
}
@tool
def recall(object_id: str, offset: int=0, limit: int=48) -> dict:
    """로컬 객체 본문의 한 페이지와 다음 참조를 반환합니다."""
    if offset<0 or not 1<=limit<=100: raise ValueError('잘못된 페이지 범위')
    obj=objects[object_id]
    end=offset+limit
    return {'id':object_id,'text':obj['text'][offset:end], 'links':obj['links'],
            'next_offset':end if end<len(obj['text']) else None}
first=recall.invoke({'object_id':'im:42'})
second=recall.invoke({'object_id':first['links'][0]})
third=recall.invoke({'object_id':second['links'][0]})
assert 'Plan B' in third['text']
parts=[]; offset=0
while True:
    page=recall.invoke({'object_id':'im:42','offset':offset})
    parts.append(page['text'])
    if page['next_offset'] is None: break
    offset=page['next_offset']
assert ''.join(parts)==objects['im:42']['text']
print(first); print(third)

{'id': 'im:42', 'text': 'IM에서 정한 목표: Plan A로 버그 수정. IM에서 정한 목표: Plan A로 버', 'links': ['worker:7'], 'next_offset': 48}
{'id': 'pr:9', 'text': 'PR 검토에서 Plan B로 변경. 회귀 테스트 추가.', 'links': [], 'next_offset': None}


## 로그에서 모델 입력을 재구성하기
DeepSeek Harness 저장소는 “Model-visible means logged” 원칙을 둡니다. 회수한 본문도 모델에 넣었다면 기록해야 과거 입력을 설명할 수 있습니다. 객체가 나중에 바뀌었을 때 ID만으로 재조회하면 과거 입력과 달라집니다. 아래에서는 실제로 넣은 payload를 복사해 기록합니다.

LangGraph checkpoint는 실행 상태를 복원하는 수단입니다. 애플리케이션의 append-only 감사 로그와 자동으로 같아지지는 않습니다. 이 예제의 이벤트 vocabulary는 교육용이며 공식 SessionEvent 형식을 구현하지 않습니다.

In [2]:
import json
from copy import deepcopy
from langchain_core.messages import SystemMessage, HumanMessage
log=[]
def append(kind,payload):
    log.append({'seq':len(log)+1,'kind':kind,'payload':deepcopy(payload)})
def project(events):
    messages=[]
    for e in events:
        if e['kind']=='user': messages.append(HumanMessage(content=e['payload']))
        elif e['kind']=='context': messages.append(SystemMessage(content=json.dumps(e['payload'],ensure_ascii=False)))
    return messages
append('user','이전 PR에서 바뀐 계획은?')
append('context',third)
request=project(log)
objects['pr:9']['text']='나중에 변경된 Plan C'
assert 'Plan B' in request[-1].content
assert [m.content for m in project(log)]==[m.content for m in request]
append('ui-only','spinner shown')
assert len(project(log))==2
print([m.content for m in project(log)])

['이전 PR에서 바뀐 계획은?', '{"id": "pr:9", "text": "PR 검토에서 Plan B로 변경. 회귀 테스트 추가.", "links": [], "next_offset": null}']


## 직접 바꿔 보기

객체 참조가 순환하거나 삭제되었을 때 recall을 어떻게 끝낼까요? 요약을 10자로 줄이면 어떤 정보를 잃는지도 관찰하세요.

## 해설과 참고 답안

방문 ID 집합, 최대 hop 수, 읽기 예산을 함께 둡니다. 삭제는 tombstone 또는 명시적 not-found로 표현합니다. 요약은 다음 객체를 찾을 ID와 작업 변경 사실을 보존해야 합니다. 전체 원문은 별도로 남겨야 합니다.